In [ ]:
# auto_stop_fulln.ipynb -- overnight sentinel. Run on EVERY pod before sleep.
# Stops training cleanly (supervisor SIGTERM -> lease self-revokes) and then
# stops THIS RunPod pod (billing ends; /workspace persists) when ANY of:
#   1. FULL-N group fully terminal (the normal finish);
#   2. STALL: FULL-N remaining has not DECREASED for STALL_HOURS -- the fleet
#      is assumed broken; every pod's sentinel sees the same shared counter,
#      so the whole fleet stops within one poll of each other;
#   3. LOCAL GRADUATION: this machine trains no n=FULL_N combo AND none is
#      claimable (the rest are live on OTHER VMs) -- our part of FULL-N is
#      over; stop now instead of burning the night on small-n (which would
#      likely be pruned to champion families tomorrow anyway). Two consecutive
#      polls required, so a lane switching combos never false-triggers.
# Interrupted combos are free: checkpoints resume, reconcile refunds attempts.
#
# FIRST TIME: on ONE pod set TEST_SHUTDOWN=True -> full dress rehearsal of the
# stop path immediately; confirm the pod shows Stopped in the console, start
# it again, re-run training.ipynb, then roll out with TEST_SHUTDOWN=False.
import copy, json, os, re, shutil, signal, socket, subprocess, sys, time
from pathlib import Path

TEST_SHUTDOWN = False
REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
POLL_SECONDS = 120
STALL_HOURS = 4          # no FULL-N progress for this long -> assume broken, stop
GRAD_POLLS = 2           # consecutive polls of local graduation before stopping

# ---- preflight: fail LOUDLY now, not at 5am --------------------------------
pod_id = os.environ.get('RUNPOD_POD_ID')
ctl = shutil.which('runpodctl')
print(f'preflight: RUNPOD_POD_ID={pod_id or "MISSING"}  runpodctl={ctl or "MISSING"}')
if not pod_id or not ctl:
    print('!! preflight FAILED: this pod cannot stop itself. Training would be')
    print('   stopped but the pod would keep billing while idle. Fix before')
    print('   sleeping, or plan to stop pods from the RunPod console.')

if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

root = Path(REPO) / OUT_DIR
cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
combos = list(cfg.iter_combos())
_counts = [int(c.train_games) for c in combos]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
fulln_ids = [c.combo_id for c in combos if int(c.train_games) == FULL_N]
HOST = socket.gethostname()
_m = re.fullmatch(r'Pod_(\d+)', Path.cwd().name)
BUNDLE_BASE = f'VM{_m.group(1)}' if _m else None
print(f'sentinel: {len(fulln_ids)} FULL-N combos; poll={POLL_SECONDS}s; '
      f'stall={STALL_HOURS}h; host={HOST}; bundle={BUNDLE_BASE or "-"}; '
      f'TEST_SHUTDOWN={TEST_SHUTDOWN}')


def _read(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception:
        return None


def _terminal(cid):
    d = root / cid
    if (d / 'done.json').exists() or (d / 'failed.json').exists():
        return True
    man = _read(d / 'vicreg_review_h5_manifest.json')
    return bool(man) and man.get('status') == 'done'


def fresh_vms():
    """name -> record for every registry lease that is still fresh."""
    now = time.time()
    out = {}
    for f in (root / 'VM_parallel').glob('*.json'):
        rec = _read(f) or {}
        if float(rec.get('expiry', 0) or 0) > now:
            out[str(rec.get('vm') or f.stem)] = rec
    return out


def my_vm_names(vms):
    mine = set()
    fam = re.compile(re.escape(BUNDLE_BASE) + r'(?:_\d+)?$') if BUNDLE_BASE else None
    for name, rec in vms.items():
        if (rec.get('info') or {}).get('host') == HOST:
            mine.add(name)
        elif fam and fam.fullmatch(name):
            mine.add(name)
    return mine


def survey():
    """(remaining_ids, i_train_fulln, fulln_claimable)"""
    vms = fresh_vms()
    mine = my_vm_names(vms)
    remaining_ids, i_train, claimable = [], False, False
    for cid in fulln_ids:
        if _terminal(cid):
            continue
        remaining_ids.append(cid)
        st = _read(root / cid / 'status.json')
        owner = st.get('vm') if st else None
        if owner in mine:
            i_train = True
        elif owner is None or owner not in vms:
            claimable = True           # free, or claimed by a dead VM
    return remaining_ids, i_train, claimable


def stop_everything(reason):
    print(f'sentinel: {reason} -- stopping training on this pod', flush=True)
    subprocess.run(['pkill', '-f', 'sweep/supervisor.py'])   # SIGTERM: lease self-revokes
    time.sleep(10)
    subprocess.run(['pkill', '-9', '-f', 'sweep/worker.py'])
    if pod_id and ctl:
        print(f'sentinel: runpodctl stop pod {pod_id}', flush=True)
        r = subprocess.run(['runpodctl', 'stop', 'pod', pod_id],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr, flush=True)
        if r.returncode != 0:
            print('!! runpodctl stop FAILED -- pod is idle but still billing; '
                  'stop it from the RunPod console.', flush=True)
    else:
        print('!! cannot self-stop: training stopped, but STOP THE POD from '
              'the console.', flush=True)


if TEST_SHUTDOWN:
    stop_everything('TEST_SHUTDOWN dress rehearsal')
    print('test issued: within ~a minute this pod should show Stopped in the '
          'RunPod console. Restart it afterwards and re-run training.ipynb '
          '(startup cleanup + attempt refund make the interruption free).')
else:
    last_n = None
    last_progress_ts = time.time()
    grad_streak = 0
    while True:
        left, i_train, claimable = survey()
        n = len(left)
        if last_n is None or n < last_n:
            last_n = n
            last_progress_ts = time.time()
        if n == 0:
            stop_everything('FULL-N group fully terminal')
            break
        if time.time() - last_progress_ts > STALL_HOURS * 3600:
            stop_everything(f'STALL: FULL-N remaining stuck at {n} for '
                            f'{STALL_HOURS}h -- assuming a fleet problem')
            break
        if not i_train and not claimable:
            grad_streak += 1
            if grad_streak >= GRAD_POLLS:
                stop_everything(f'LOCAL GRADUATION: this VM trains no FULL-N combo '
                                f'and none is claimable ({n} remain, all live on '
                                f'other VMs)')
                break
        else:
            grad_streak = 0
        stall_min = (time.time() - last_progress_ts) / 60
        print(f'{time.strftime("%H:%M:%S")} FULL-N remaining: {n} '
              f'(training-fulln-here={i_train} claimable={claimable} '
              f'no-progress={stall_min:.0f}m grad-streak={grad_streak})', flush=True)
        time.sleep(POLL_SECONDS)
